# 🔌 3. MCP & Skills: 데이터 소스를 서버로 분리하고, 에이전트가 스스로 쓰게 하기

1교시의 에이전트는 검색 도구를 **에이전트 코드 안에 직접** 바인딩해서 사용했습니다.
하지만 엔터프라이즈 환경에서 데이터 소스가 수십 개로 늘어나고 각 데이터베이스의 소유권(Governance)과 배포 주기가 나뉘면, 도구를 에이전트 코드에 직접 하드코딩하는 방식은 즉각 한계에 직면합니다.

이번 시간에는 데이터 접근 계층을 독립적인 마이크로서비스인 **MCP (Model Context Protocol) 서버**로 분리하고,
같은 MCP 서버를 에이전트에 연결하는 **세 가지 방식(정적 바인딩 · Search & Proxy · Skills)** 을 직접 조립해 비교합니다.

---

### 🎯 학습 목표
1. **FastMCP 프레임워크**로 MCP 표준 서버를 구축하고, Pydantic 스키마를 통해 LLM이 생성한 비정상 인자를 서버 레벨에서 엄격히 차단합니다.
2. **Bearer 토큰 인증**과 **두 층의 인가(scope로 도메인 접근, tag × claim으로 데이터 등급)** 를 적용하여, 같은 서버라도 사용자에 따라 쓸 수 있는 도구가 달라지도록 통제합니다.
3. 동일한 MCP 서버를 **정적 바인딩(`MultiServerMCPClient`)**, **Search & Proxy(검색 도구 + 범용 호출 도구 + Middleware)**, **Skills 기반 점진적 공개(Progressive Disclosure)** 세 방식으로 각각 조립해 보고, 컨텍스트 사용량 · 호출 경로 · 확장성 관점에서 트레이드오프를 비교합니다.

### 🎓 학습 목차

| 파트 | 주제 | 핵심 내용 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | 프로젝트 루트 탐색, `.env`, `nest_asyncio` |
| **Part 1** | 왜 MCP인가 (아키텍처 배경) | 데이터 접근 계층 분리, MCP 3대 Primitives, 도구 폭발 문제 |
| **Part 2** | FastMCP 서버 구축 & 스키마 검증 | 파이썬 데코레이터(`@mcp.tool`), Pydantic 검증, In-Process 테스트 |
| **Part 3** | 엔터프라이즈 인증/인가 (Auth) | Bearer Token, 역할별 토큰, scope · tag · claim 권한 체계, `AuthContext` 등급 규칙 |
| **Part 4** | 연결 방식 A: 정적 바인딩 | `langchain_mcp_adapters`로 일괄 로드 및 한계(Context Bloat) |
| **Part 5** | 연결 방식 B: Search & Proxy | 도구 카탈로그, `search_mcp_tools` + `call_mcp_tool`, `wrap_tool_call` 정책 계층 |
| **Part 6** | 연결 방식 C: Skills 기반 점진적 공개 | 범용 도구 3종, 스킬 카탈로그 주입, `--token-env` 보안, 자율 탐색 |
| **Part 7** | 정리와 Mission 03 | 세 방식 비교, 프로덕션 서버 구현 및 두 에이전트 대화 비교 |

> 📌 **진행 순서**: Part 1\~2 & Part 3(이론) → **Mission 02** (MCP 서버 구현·기동, `missions/02_missions.md`) → Part 3(실습)\~Part 6.1 (내가 만든 서버로 실행) → **Mission 03** (CLI 검증 · `mcp_servers.json` 등록 · Chainlit 비교, `missions/03_missions.md`)

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 프로젝트 루트 경로 자동 설정 (app/ 폴더가 있는 위치까지 상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 로드 (프로젝트 루트의 .env)
load_dotenv(os.path.join(project_root, ".env"), override=True)

print(f"✅ Working Directory: {os.getcwd()}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. 공통 유틸리티 로드
from app.utils import init_chat_model, normalize_content

import logging
logging.getLogger("fastmcp").setLevel(logging.CRITICAL)                 # 서버 측 검증 오류 로그는 출력하지 않음
logging.getLogger("langchain_google_genai").setLevel(logging.ERROR)     # 스키마 변환 경고 생략

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)

---
## 🧭 Part 1. 왜 MCP인가: 데이터 접근 계층의 분리와 개방형 표준

### 1.1 도구 직접 바인딩의 한계 vs MCP 마이크로서비스 아키텍처

1교시의 에이전트는 데이터베이스 접속 객체와 검색 로직을 에이전트 프로세스 내부에 직접 품고 있었습니다. 이는 프로토타입 단계에서는 편리하지만, 실무 환경에서는 다음과 같은 심각한 병목을 만듭니다:

| 비교 항목 | 도구를 에이전트 코드에 직접 포함 (Monolithic) | MCP 서버로 분리 (Microservice) |
|:---|:---|:---|
| **소유와 배포 주기** | 데이터 엔지니어가 검색 로직을 수정할 때마다 **에이전트 전체를 재배포**해야 함 | **데이터 팀이 MCP 서버만 독립 배포**. 에이전트는 무중단 서비스 유지 |
| **권한 및 보안** | 에이전트 실행 런타임이 모든 DB 접속 키와 원천 데이터를 직접 보유함 | **MCP 서버가 인증·인가 게이트웨이 역할** 수행. 에이전트는 토큰만 소유 |
| **크로스 플랫폼 재사용** | 에이전트마다 동일한 DB 검색 도구 코드를 중복 구현/복사해야 함 | 사내 다양한 에이전트, 웹 서비스, Claude Desktop 등이 **단일 서버를 표준 프로토콜로 공유** |
| **언어 및 런타임 격리** | 에이전트 런타임(Python)과 동일한 환경으로 도구 구현이 제한됨 | MCP 서버는 Node.js, Go, Rust 등 **데이터 소스에 최적화된 언어로 구축 가능** |

---

### 1.2 Model Context Protocol (MCP)이란?

**MCP(Model Context Protocol)** 는 2024년 11월 Anthropic이 오픈소스로 공개한 **AI 애플리케이션과 외부 데이터 소스/도구 간의 범용 개방형 표준 프로토콜**입니다. 하드웨어의 USB-C 포트처럼, 모델과 데이터 간의 표준화된 규격을 정의합니다.

#### 📌 MCP의 3대 핵심 Primitives (기본 구성요소)
1. **Tools (도구)**:
   - 모델이 계산, 외부 API 호출, DB 쿼리 등 **부수 효과(Side-effect)**를 일으키며 실행할 수 있는 원격 RPC 함수입니다.
   - Pydantic 또는 JSON Schema를 통해 매개변수 규격을 엄격히 정의합니다.
2. **Resources (리소스)**:
   - 파일 내용, 데이터베이스 테이블 스키마, 시스템 로그 등 **읽기 전용(Read-only) 데이터 컨텍스트**를 제공하는 파일/URI 인터페이스입니다. (GET 요청과 유사)
3. **Prompts (프롬프트)**:
   - 서버가 클라이언트에게 제공하는 **사전 정의된 슬래시 커맨드 및 재사용 가능한 프롬프트 템플릿**입니다.

#### 🌐 전송 방식 (Transports)의 진화: SSE에서 Stateless Streamable HTTP로
- **`stdio` (표준 입출력)**: 로컬 CLI 환경이나 데스크톱 앱(Claude Desktop 등)에서 하위 프로세스를 직접 띄워 표준 입출력 파이프로 통신할 때 사용합니다.
- **`Streamable HTTP` (HTTP 방식)**: 과거의 SSE(Server-Sent Events) 방식을 대체하여, 최신 MCP 사양과 FastMCP는 단일 엔드포인트(예: `/mcp`)에 표준 HTTP POST 요청을 보내는 **Stateless(무상태) Streamable HTTP**를 기본으로 사용합니다. 요청마다 세션 없이 독립적으로 처리되므로 로드밸런서 뒤에서 자유롭게 수평 확장(Auto-scaling)할 수 있으며, 응답은 표준 스트리밍 형식으로 안전하게 전달됩니다.

---

### 1.3 새로운 과제: 도구 폭발 (Tool Explosion)
MCP로 데이터 소스를 마이크로서비스화하면 접근성이 비약적으로 높아지지만, 서버가 10개, 도구가 100개로 늘어나면 **"100개의 도구 스키마를 어떻게 에이전트에게 전달할 것인가?"**라는 새로운 아키텍처 문제가 발생합니다.
이 문제는 Part 4(정적 바인딩), Part 5(Search & Proxy), Part 6(Skills 점진적 공개)에서 세 가지 연결 방식으로 비교합니다.

---
## 🛠️ Part 2. FastMCP 서버 구축 & Pydantic 스키마 검증

**FastMCP**는 FastAPI 스타일의 직관적인 데코레이터(`@mcp.tool`, `@mcp.resource`)를 사용하여 수십 줄의 보일러플레이트 코드 없이 파이썬 함수를 즉시 MCP 표준 서버로 만들어 주는 고수준 프레임워크입니다.

먼저 네트워크 포트 바인딩 없이, **같은 Python 프로세스 메모리 내(In-Process)**에서 서버와 클라이언트를 연결하여 동작 원리와 스키마 검증 메커니즘을 확인합니다.

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field
from fastmcp import FastMCP, Client
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

logging.getLogger("fastmcp").setLevel(logging.CRITICAL)   # fastmcp는 import 시 로그 레벨을 다시 설정하므로 import 뒤에 지정

policy_store = Chroma(
    collection_name="enterprise_policy_store",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-large"),
    persist_directory="app/database/chroma_db",
)

demo_server = FastMCP(name="policy-demo", instructions="사내 규정을 검색하는 데모 서버")


@demo_server.tool(description="사내 규정을 검색하여 관련 조항을 반환합니다.")
def search_policy(query: str) -> str:
    docs = policy_store.similarity_search(query, k=1)
    return docs[0].metadata["breadcrumb"] + "\n" + docs[0].metadata["raw_content"] if docs else "결과 없음"


# Pydantic 스키마: LLM이 잘못된 인자를 보내면 서버에서 거부됩니다.
class Grade(str, Enum):
    EXECUTIVE = "임원"
    LEADER = "팀장"
    MEMBER = "팀원"


class AllowanceQuery(BaseModel):
    grade: Grade = Field(description="직급 구분")
    days: int = Field(ge=1, le=30, description="출장 일수 (1~30)")


@demo_server.tool(description="직급과 출장 일수로 일비 총액을 계산합니다. (제15조 기준)")
def calc_daily_allowance(query: AllowanceQuery) -> str:
    per_day = {"임원": 40000, "팀장": 30000, "팀원": 25000}[query.grade.value]
    return f"{query.grade.value} {query.days}일 일비: {per_day * query.days:,}원"

In [ ]:
async def demo_in_process1():
    async with Client(demo_server) as client:
        tools = await client.list_tools()
        print("📋 도구 목록:", [t.name for t in tools])

await demo_in_process1()

In [ ]:
async def demo_in_process2():
    async with Client(demo_server) as client:
        try:
            r = await client.call_tool("calc_daily_allowance", {"query": {"grade": "팀장", "days": 3}})
            print("✅ 정상 호출:", r.data)
        except Exception as e:
            print("❌ 스키마 위반 차단:", type(e).__name__)

        try:
            r = await client.call_tool("calc_daily_allowance", {"query": {"grade": "과장", "days": 99}})
            print("✅ 정상 호출:", r.data)
        except Exception as e:
            print("❌ 스키마 위반 차단:", type(e).__name__)

await demo_in_process2()

In [ ]:
async def demo_in_process3():
    async with Client(demo_server) as client:
        r = await client.call_tool("search_policy", {"query": "정산 신청 기한"})
        print("\n✅ 규정 검색:\n", r.data)

await demo_in_process3()

> ### 🛡️ LLM과 MCP 백엔드 사이의 방어선
> LLM은 확률적 모델이므로, 사내 규정에 존재하지 않는 직급(`"과장"`)이나 물리적으로 불가능한 값(`99일`)을 도구 인자로 보낼 수 있습니다.
> 
> 1. FastMCP 서버는 MCP 도구 입력의 표준 **JSON Schema**를 Pydantic 모델과 파이썬 타입 힌트를 통해 자동 생성하고, 이를 LLM에게 도구 스펙으로 제공합니다.
> 2. LLM이 호출을 시도할 때, FastMCP는 비즈니스 로직을 실행하기 전에 **Pydantic 유효성 검증을 먼저 수행(Fail-fast)**합니다.
> 3. 규격을 위반한 요청은 즉시 차단되어 DB 과부하나 쿼리 에러를 미연에 방지합니다.
>
> 또한 `Client(demo_server)`와 같은 **In-Process 클라이언트**는 복잡한 포트 바인딩 없이도 단위 테스트(Unit Test)를 빠르게 수행할 수 있는 강력한 도구입니다.

---
## 🔐 Part 3. 엔터프라이즈 인증/인가 (Auth) 아키텍처

사내 규정, 재무 지표, 프로젝트 예산 등 기업 데이터는 모든 에이전트에게 무제한으로 열려 있어서는 안 됩니다. 데이터 접근 계층을 마이크로서비스로 분리했다면, **반드시 접근 제어(Access Control)가 동반**되어야 합니다.

`app/mcp/enterprise_rag_server.py`는 두 단계로 접근을 통제합니다.
1. **인증**: 등록된 Bearer 토큰이 있어야 서버에 들어올 수 있습니다.
2. **도구별 인가**: 들어온 뒤에도 토큰의 권한(scope)과 사용자 속성(clearance)에 따라 **쓸 수 있는 도구가 달라집니다.**

### 🛡️ 인증과 두 층의 인가

| 보안 계층 | 질문 | FastMCP 구현 | 실패 시 동작 |
|:---|:---|:---|:---|
| **인증** | *"등록된 토큰인가?"* | `FastMCP(..., auth=verifier)` | `401 Unauthorized` |
| **인가 1층: scope** | *"이 데이터 도메인에 접근할 수 있는가?"* | 도구마다 `@mcp.tool(auth=require_scopes("erp:read"))` | 도구가 목록에서 숨겨지고 호출 거부 |
| **인가 2층: 등급** | *"이 등급의 데이터를 볼 수 있는가?"* | 서버 전역 `AuthMiddleware(auth=require_clearance)` | 도구가 목록에서 숨겨지고 호출 거부 |

---

#### 1. 권한을 세 축으로 나눠 이름 짓기

문서 종류마다 권한 이름을 만들면(`report:read`, `guide:read`, ...) 문서가 늘어날 때마다 권한도 늘어나 관리할 수 없습니다. 그래서 **성격이 다른 세 가지를 분리**합니다.

| 축 | 무엇을 나타내나 | 이 서버의 값 | 개수의 기준 |
|:---|:---|:---|:---|
| **scope** (토큰) | 클라이언트가 **어떤 시스템에** 접근할 수 있나 | `docs:read` `erp:read` `graph:read` (`<도메인>:<동작>`) | 시스템 수 |
| **tag** (도구) | 도구가 다루는 **데이터의 성격** | `data:docs` · `data:erp` · `data:graph`, `class:public` \~ `class:secret` | 분류 체계 |
| **claim** (토큰) | 요청한 **사람의 속성** | `role`, `department`, `clearance` | 조직 구조 |

`class:*` 등급은 실습 데이터의 **정보보안관리지침 4등급**(공개 `public` < 사내한 `internal` < 대외비 `confidential` < 극비 `secret`)을 그대로 씁니다.

| 도구 | tag | scope |
|:---|:---|:---|
| `search_company_policy` (사내 규정) | `data:docs` `class:internal` | `docs:read` |
| `search_aws_guides` (AWS 백서) | `data:docs` `class:public` | `docs:read` |
| `search_enterprise_erp` (사내 ERP) | `data:erp` `class:confidential` | `erp:read` |
| `graph_*` 탐색 도구 5종 (지식 그래프) | `data:graph` `class:confidential` | `graph:read` |

역할별 토큰 (`StaticTokenVerifier`에 등록한 dict 전체가 claims가 됩니다):

| 토큰 | role | clearance | scopes |
|:---|:---|:---|:---|
| `<기본 토큰>` (에이전트 서비스 계정) | `service` | `confidential` | `docs:read` `erp:read` `graph:read` |
| `<기본 토큰>:partner` (외부 협력사) | `partner` | `public` | `docs:read` |
| `<기본 토큰>:staff` (일반 직원) | `staff` | `internal` | `docs:read` |
| `<기본 토큰>:manager` (팀장) | `manager` | `confidential` | `docs:read` `erp:read` |
| `<기본 토큰>:executive` (임원) | `executive` | `secret` | `docs:read` `erp:read` `graph:read` |

> 운영 환경에서는 IdP가 사용자마다 JWT를 발급하고, role · clearance 같은 정보가 **JWT claim**으로 들어옵니다. 인가 규칙 코드는 그대로입니다.

---

#### 2. 등급 규칙: `AuthContext`로 만든 사용자 정의 함수

`auth=`에는 **`AuthContext`(검증된 토큰 + 접근하려는 도구)를 받아 True/False를 돌려주는 함수**를 넣습니다. `require_scopes`도 FastMCP가 미리 만들어 둔 이런 함수입니다.
등급 규칙은 **도구의 tag(`ctx.component.tags`)와 사용자의 claim(`ctx.token.claims`)을 비교**합니다.

```python
CLEARANCE_LEVELS = ["public", "internal", "confidential", "secret"]

def require_clearance(ctx: AuthContext) -> bool:
    tool_class = next((t.split(":", 1)[1] for t in ctx.component.tags if t.startswith("class:")), None)
    if tool_class not in CLEARANCE_LEVELS:      # 등급이 없는 도구는 막는다 (fail-closed)
        return False
    user_clearance = ctx.token.claims.get("clearance", "public")
    return CLEARANCE_LEVELS.index(tool_class) <= CLEARANCE_LEVELS.index(user_clearance)

mcp = FastMCP(..., middleware=[AuthMiddleware(auth=require_clearance)])
```

이 규칙을 **서버 전역 `AuthMiddleware`** 에 걸었기 때문에, 새 도구는 `tags={"class:confidential"}`처럼 **등급 태그만 붙이면 같은 정책이 자동 적용**됩니다. 문서가 아무리 많아져도 권한 이름은 늘어나지 않습니다.

> [!NOTE]
> **`AuthContext`로 판단할 수 없는 것**: `AuthContext`에는 **도구 인자 값이 들어 있지 않습니다.** "자기 부서 예산만 조회 가능"처럼 인자나 결과를 보고 판단해야 하는 규칙은 도구 함수 안에서 `get_access_token().claims`를 읽어 처리합니다.
>
> **403은 언제 나오나요?** MCP 사양은 scope가 부족한 요청에 `403 Forbidden` + `insufficient_scope`를 권장합니다. FastMCP에서는 서버 전체에 필요한 scope를 걸었을 때(토큰 검증기의 `required_scopes`) HTTP 단계에서 403이 나갑니다. 위처럼 **도구 단위**로 건 규칙은 MCP 메시지 단계에서 처리되어, 권한 없는 도구는 처음부터 보이지 않습니다.

---
## ⏸️ 잠깐! 서버를 기동하기 전에 Mission 02를 먼저 완료하세요

지금부터 실행할 코드 셀들은 **여러분이 직접 완성한 MCP 서버**(`app/mcp/enterprise_rag_server.py`, 포트 8010)를 사용합니다.  
`missions/02_missions.md`를 열어 **1\~4단계(Mission 02: 인증·인가 프로바이더 및 4대 DB 도구 등록)**를 먼저 수행하고 돌아오세요.

| 순서 | 할 일 | 위치 |
|:---|:---|:---|
| ① | Part 1\~2 & Part 3(이론): MCP 개념, FastMCP 스키마 검증, 인증/인가 2층 구조 이해 | 이 노트북 (완료) |
| ② | **🎯 Mission 02**: 토큰 확인 → `enterprise_rag_server.py` TODO 1·2·4·5 완성 → 서버 기동 확인 | `missions/02_missions.md` |
| ③ | Part 3(실습)\~Part 6.1: 역할별 가시성 검증, 정적 바인딩, Search & Proxy, Skills 개념 (**내가 만든 서버로 실행**) | 이 노트북 |
| ④ | **🎯 Mission 03**: CLI 보안 검증 → `mcp_servers.json` 등록 → Chainlit에서 `skill_rag_agent` vs `tool_rag_agent` 비교 | `missions/03_missions.md` |

> 💡 Mission 02 4단계에서 터미널에 띄운 서버(포트 8010)가 실행 중이면 아래 셀은 그 서버를 그대로 사용하고, 꺼져 있으면 자동으로 백그라운드에서 띄웁니다.  
> TODO가 남아 있어 서버가 뜨지 않으면 셀이 오류와 함께 멈춥니다. 먼저 아래 두 셀로 내가 작성한 코드와 서버 기동을 확인합니다.

In [ ]:
import re

src = open("app/mcp/enterprise_rag_server.py", encoding="utf-8").read()
start = src.find("# 1. 인증기") if "# 1. 인증기" in src else src.find("SERVER_TOKEN")
if start == -1 or "@mcp.tool(" not in src:
    print("⚠️ 인증기(StaticTokenVerifier) 또는 @mcp.tool 코드를 찾지 못했습니다.")
    print("   Mission 02(missions/02_missions.md)의 2~3단계를 완료했는지 확인하세요.")
else:
    # 인증기 · 사용자 정의 규칙 · 서버 생성 부분
    print(src[start:src.find("DB_DIR =")])
    print("...")
    # 도구별 인가 규칙 (주석 + @mcp.tool 데코레이터)
    for block in re.findall(r"(# 5-\d\..*?\n@mcp\.tool\(.*?\n\))", src, re.S):
        print(block, end="\n\n")

In [ ]:
import subprocess
import time
import httpx

MCP_URL = "http://localhost:8010/mcp"

def is_server_up(url: str) -> bool:
    try:
        httpx.get(url, timeout=2)   # 인증 서버이므로 401이어도 '떠 있음'
        return True
    except httpx.HTTPError:
        return False

mcp_process = None
if is_server_up(MCP_URL):
    print("ℹ️ 이미 실행 중인 MCP 서버를 사용합니다:", MCP_URL)
else:
    os.makedirs("artifacts", exist_ok=True)
    mcp_process = subprocess.Popen(
        [sys.executable, "app/mcp/enterprise_rag_server.py", "--port", "8010"],
        stdout=open("artifacts/mcp_server.log", "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        if is_server_up(MCP_URL) or mcp_process.poll() is not None:   # 기동 완료 또는 프로세스 종료
            break
        time.sleep(1)

    if not is_server_up(MCP_URL):
        if mcp_process.poll() is None:
            mcp_process.terminate()
        mcp_process = None
        log_tail = open("artifacts/mcp_server.log", encoding="utf-8", errors="replace").read().strip().splitlines()[-3:]
        print("\n".join(log_tail))
        raise RuntimeError(
            "❌ MCP 서버를 기동하지 못했습니다. Mission 02(missions/02_missions.md)를 먼저 완료하세요. "
            "(전체 로그: artifacts/mcp_server.log)"
        )
    print(f"🚀 MCP 서버 기동 완료 (PID {mcp_process.pid}) → {MCP_URL}")

In [ ]:
from fastmcp.client.auth import BearerAuth

TOKEN = os.environ["ENTERPRISE_RAG_MCP_TOKEN"]   # 에이전트용 기본 토큰 (값은 출력하지 않습니다)
ROLE_TOKENS = {role: f"{TOKEN}:{role}" for role in ["partner", "staff", "manager", "executive"]}   # 역할별 테스트 토큰


async def visible_tools(token):
    async with Client(MCP_URL, auth=BearerAuth(token)) as client:
        return [t.name for t in await client.list_tools()]


# 역할별로 tools/list에 보이는 도구를 표로 비교합니다.
all_tools = await visible_tools(TOKEN)
visible = {role: await visible_tools(tok) for role, tok in ROLE_TOKENS.items()}

print(f"{'MCP 도구':<26}" + "".join(f"{role:^11}" for role in ROLE_TOKENS))
for name in all_tools:
    print(f"{name:<26}" + "".join(f"{'✅' if name in visible[role] else '❌':^10}" for role in ROLE_TOKENS))

#### 3. 역할별로 보이는 도구

같은 서버에 접속해도 토큰에 따라 **`tools/list` 결과 자체가 달라집니다.** 두 층이 각각 어디서 걸러 내는지 보세요.

- **partner**: `docs:read`가 있어 scope 층은 통과하지만, clearance가 `public`이라 **사내 규정(`class:internal`)은 등급 층에서** 걸러집니다. 공개 등급인 AWS 백서만 보입니다.
- **staff**: clearance `internal`이라 사내 규정까지 보입니다. `erp:read`, `graph:read`가 없어 **ERP와 지식 그래프는 scope 층에서** 걸러집니다.
- **manager**: `erp:read`와 clearance `confidential`이 있어 ERP까지 보입니다. `graph:read`가 없어 지식 그래프는 보이지 않습니다.
- **executive**: 네 도구가 모두 보입니다.

에이전트 입장에서는 **권한 없는 도구가 아예 존재하지 않는 것**과 같습니다. 정적 바인딩(Part 4)으로 도구를 불러오면 partner 토큰의 에이전트는 도구 1개만 받게 됩니다.

그렇다면 도구 이름을 알고 **목록을 건너뛰어 직접 호출**하면 어떻게 될까요?

In [ ]:
async def try_call(role, tool, args):
    async with Client(MCP_URL, auth=BearerAuth(ROLE_TOKENS[role])) as client:
        try:
            result = await client.call_tool(tool, args)
            print(f"✅ {role:<9} → {tool}\n   {str(result.data).splitlines()[0][:80]}")
        except Exception as e:
            print(f"❌ {role:<9} → {tool}\n   {str(e).splitlines()[0][:100]}")


await try_call("partner", "search_company_policy", {"query": "출장 일비 기준"})                  # 등급 층: public < internal
await try_call("staff",   "search_enterprise_erp", {"query": "AgenticAI팀 프로젝트 예산"})        # scope 층: erp:read 없음
await try_call("manager", "search_enterprise_erp", {"query": "AgenticAI팀 프로젝트 예산"})        # 허용 (Text2SQL 실행)

- **partner → 사내 규정**: scope는 있지만 **등급 층**(서버 전역 `AuthMiddleware`)에서 *"insufficient permissions"* 로 거부됩니다.
- **staff → ERP 조회**: **scope 층**(도구의 `auth=`)에서 *"not found or not authorized"* 로 거부됩니다. 권한이 없는 쪽에는 그 도구가 있는지조차 알려 주지 않습니다.
- **manager → ERP 조회**: 두 층을 모두 통과해 Text2SQL 결과(`[status: ok]`, 실행 SQL, 결과 표)가 돌아옵니다.

거부하는 층이 다르면 메시지도 다릅니다. 그리고 모든 거부는 **서버에서** 일어납니다. 에이전트의 프롬프트를 조작하거나 도구 이름을 추측해 호출해도 토큰의 권한을 넘을 수 없습니다.

#### 4. 클라이언트는 토큰을 어떻게 보내나요? (`BearerAuth`)

FastMCP의 `BearerAuth`는 HTTP 요청 헤더에 `Authorization: Bearer <토큰>`을 붙이는 역할만 합니다.
```python
# fastmcp 내부 BearerAuth 구현 (요약)
class BearerAuth(httpx.Auth):
    def __init__(self, token: str):
        self.token = SecretStr(token)

    def auth_flow(self, request):
        request.headers["Authorization"] = f"Bearer {self.token.get_secret_value()}"
        yield request
```
토큰이 없거나 등록되지 않은 값이면 인가 단계까지 가지 못하고 `401 Unauthorized`로 끝납니다.

> [!NOTE]
> **실무 환경(Production)으로의 확장:**
> 본 실습의 `StaticTokenVerifier`와 `:staff` 같은 역할 토큰은 개발 · 시연용입니다.
> 운영에서는 사내 통합 인증 시스템(Keycloak, Microsoft Entra ID, Okta 등)이 발급한 JWT를 **`JWTVerifier`** 로 검증하고, role · clearance는 JWT claim에서 옵니다. `require_scopes`, `require_clearance`, `AuthMiddleware` 같은 **인가 규칙 코드는 그대로** 재사용됩니다.
> 문서가 수백만 건 단위인 조직은 문서 하나하나에 권한을 붙이지 않습니다. 원천 시스템의 폴더 · 스페이스에서 **등급과 소유 조직을 상속**받고, 대규모라면 관계 기반 권한(ReBAC, Google Zanzibar 계열의 OpenFGA · SpiceDB)을 씁니다.

---

---
## 🔗 Part 4. 연결 방식 A: 정적 바인딩 (`langchain_mcp_adapters`)

이제 구축된 MCP 서버를 에이전트에 연결하는 방법을 살펴봅니다.
가장 먼저 떠오르는 직관적인 방법은, 에이전트 시작 시점에 MCP 서버의 모든 도구를 긁어와(Fetch) LangChain 도구로 변환한 뒤 `create_agent`에 직접 주입하는 **정적 바인딩(Static Binding)** 방식입니다.

`langchain_mcp_adapters` 라이브러리의 `MultiServerMCPClient`를 사용하면 원격 MCP 서버의 HTTP 엔드포인트와 인증 헤더를 설정하여 도구들을 한 번에 로드할 수 있습니다.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

mcp_client = MultiServerMCPClient({
    "enterprise-rag": {
        "transport": "streamable_http",
        "url": MCP_URL,
        "headers": {"Authorization": f"Bearer {TOKEN}"},
    }
})
mcp_tools = await mcp_client.get_tools()
print("📦 정적 바인딩된 도구:", [t.name for t in mcp_tools])

`langchain_mcp_adapters`는 원격 MCP 서버의 도구 스펙(JSON Schema)을 LangChain 표준 `StructuredTool` 객체로 자동 변환해 주는 어댑터입니다. 변환된 도구 객체를 확인해 봅니다.

In [ ]:
mcp_tools

In [ ]:
static_agent = create_agent(
    model=llm,
    tools=mcp_tools,
    system_prompt="당신은 사내 데이터 Q&A 에이전트입니다. 도구 결과에 근거해서만 답하고 출처를 명시하세요.",
    checkpointer=InMemorySaver(),
)

async def ask(agent, question, thread_id):
    result = await agent.ainvoke(
        {"messages": [HumanMessage(content=question)]},
        {"configurable": {"thread_id": thread_id}},
    )
    last_human = max(i for i, m in enumerate(result["messages"]) if m.type == "human")
    calls = [tc for m in result["messages"][last_human:] for tc in (getattr(m, "tool_calls", None) or [])]
    for i, tc in enumerate(calls, 1):
        print(f"  🔄 Step {i}: {tc['name']}({str(tc['args'])[:160]})")
    print(f"  ➜ 도구 호출 {len(calls)}회\n")
    print("💬", normalize_content(result["messages"][-1].content)[:1200])
    return result

# 세 연결 방식(Part 4·5·6)에 같은 질문을 던져 호출 경로를 비교합니다.
COMPARE_QUESTION = "정보보안관리지침상 대외비 데이터나 소스코드를 사외로 반출할 때의 승인 절차와 조건은?"

_ = await ask(static_agent, COMPARE_QUESTION, "static-1")

### 💡 정적 바인딩의 한계: 도구 폭발 (Tool Explosion Problem)

정적 바인딩은 코드가 직관적이고 구현이 단순하지만, 데이터 소스가 늘어나는 순간 다음과 같은 심각한 한계에 부딪힙니다:

| 문제점 | 실무적 영향 및 원인 |
|:---|:---|
| **Context Bloat (토큰 낭비)** | 사내에 10개 MCP 서버가 있고 서버당 10개의 도구가 있다면, **100개 도구의 방대한 JSON Schema 전체가 매 대화 턴마다 시스템 프롬프트에 주입**됩니다. 질문 1개를 처리하는 데 기본 컨텍스트만 수만 토큰을 소모합니다. |
| **Tool Confusion (도구 환각 & 오선택)** | 프롬프트에 도구 설명서가 100개씩 나열되어 있으면, LLM의 주의 집중(Attention)이 분산되어 유사한 이름의 엉뚱한 도구를 호출하거나 엉뚱한 매개변수를 생성할 확률이 급격히 증가합니다. |
| **강한 결합과 재배포 비용 (Tight Coupling)** | 사내에 새로운 부서의 MCP 서버가 생기거나 기존 서버의 도구 스펙이 변경될 때마다, **에이전트 코드(`MultiServerMCPClient` 설정)를 직접 수정하고 에이전트 서비스를 재배포**해야 합니다. |

> [!WARNING]
> 즉, **"모든 도구를 미리 프롬프트에 쏟아붓는 방식"**은 엔터프라이즈 규모로 확장할 수 없습니다.
>
> 그렇다면 매 턴 질문에 맞는 도구만 골라 `tools`를 바꿔 끼우면 될까요? 이 방법은 **프롬프트 캐싱을 깨뜨립니다.** 캐싱은 프롬프트 앞부분(prefix)이 완전히 같아야 적중하는데, 도구 정의는 그 prefix의 맨 앞(`tools → system → messages`)에 놓이기 때문입니다.
>
> 그래서 실무 해법은 **"`tools`는 고정하고, 필요한 도구 정보는 대화 뒤에 덧붙인다"** 는 원칙을 따릅니다. 이 원칙을 구현하는 두 가지 방식을 Part 5(**Search & Proxy**)와 Part 6(**Skills**)에서 살펴봅니다.

---
## 🔎 Part 5. 연결 방식 B: Search & Proxy

정적 바인딩은 MCP 도구 전부를 `tools`에 넣었습니다. Search & Proxy 방식은 반대로, **MCP 도구를 하나도 바인딩하지 않습니다.**
대신 에이전트에게는 **범용 도구 2개만 고정**으로 쥐여 줍니다.

1. **`search_mcp_tools`**: 도구 카탈로그에서 필요한 도구를 검색해 **이름 · 설명 · 입력 스키마**를 돌려줍니다.
2. **`call_mcp_tool`**: 검색으로 찾은 도구의 이름과 인자를 받아 **MCP 서버에 대신 호출(Proxy)** 합니다.

```mermaid
flowchart LR
    subgraph Agent ["(create_agent) — tools 2개 고정"]
        S["search_mcp_tools"]
        C["call_mcp_tool"]
    end
    Cat["도구 카탈로그<br/>(파이썬 메모리, LLM에 미바인딩)"]
    MW["DiscoveredToolGuard<br/>(wrap_tool_call)"]
    MCP["Enterprise-RAG-Server (:8010)"]

    S -->|"1. 키워드 검색"| Cat
    Cat -->|"2. 이름·설명·스키마를<br/>대화 뒤에 덧붙임"| Agent
    C -->|"3. 실행 직전 검사"| MW
    MW -->|"4. 허용된 호출만 전달"| MCP
```

먼저 MCP 서버의 도구 목록을 읽어 **도구 카탈로그**를 만듭니다. 카탈로그는 에이전트 프롬프트가 아니라 파이썬 메모리에 있으므로, 도구가 100개로 늘어나도 프롬프트는 변하지 않습니다.

In [ ]:
import json

# 도구 카탈로그: MCP 서버의 도구 스펙을 에이전트 '밖'(파이썬 메모리)에 보관합니다. LLM에는 바인딩하지 않습니다.
async def load_tool_catalog():
    async with Client(MCP_URL, auth=BearerAuth(TOKEN)) as client:
        return {
            t.name: {"description": t.description, "input_schema": t.inputSchema}
            for t in await client.list_tools()
        }

TOOL_CATALOG = await load_tool_catalog()
print(f"🗂️ 카탈로그에 등록된 MCP 도구 {len(TOOL_CATALOG)}개:", list(TOOL_CATALOG))

### 5.1 고정된 두 개의 범용 도구

| 도구 | 역할 | 반환값 |
|:---|:---|:---|
| `search_mcp_tools(query)` | 카탈로그에서 키워드로 도구를 검색 | 매칭된 도구의 **이름 · 설명 · 입력 스키마(JSON)** |
| `call_mcp_tool(name, arguments)` | 검색으로 찾은 도구를 MCP 서버에 대신 호출 (Proxy) | MCP 도구의 실행 결과 |

검색 결과는 **도구 결과(ToolMessage)로 대화 기록 뒤쪽에 덧붙습니다.** 모델은 이 스키마를 읽고 `call_mcp_tool`의 `arguments`를 채웁니다.
`arguments`는 shell 명령 문자열이 아니라 **구조화된 JSON 객체**로 그대로 전달되므로, 따옴표 이스케이프 같은 문제가 생기지 않습니다.

> 💡 검색은 교육용으로 단순한 키워드 매칭을 사용합니다. 실무에서는 도구 설명을 임베딩해 의미 검색(Dense)이나 BM25로 바꾸는 경우가 많습니다.

In [ ]:
from langchain.tools import tool


@tool
def search_mcp_tools(query: str) -> str:
    """사내 MCP 서버에 등록된 도구를 키워드로 검색하여 이름, 설명, 입력 스키마(JSON Schema)를 반환합니다.
    도구를 실행하기 전에 반드시 먼저 검색하세요. 키워드는 공백으로 구분합니다."""
    keywords = query.lower().split()
    scored = []
    for name, spec in TOOL_CATALOG.items():
        text = f"{name} {spec['description']}".lower()
        score = sum(k in text for k in keywords)
        if score:
            scored.append((score, name))
    top = [name for _, name in sorted(scored, reverse=True)[:3]]
    return json.dumps(
        {"results": [{"name": n, **TOOL_CATALOG[n]} for n in top], "all_tool_names": list(TOOL_CATALOG)},
        ensure_ascii=False,
    )


@tool
async def call_mcp_tool(name: str, arguments: dict) -> str:
    """search_mcp_tools로 찾은 MCP 도구를 실행합니다. arguments는 검색 결과의 input_schema를 따르는 JSON 객체입니다."""
    async with Client(MCP_URL, auth=BearerAuth(TOKEN)) as client:
        result = await client.call_tool(name, arguments, raise_on_error=False)
    text = "\n".join(c.text for c in result.content if hasattr(c, "text"))
    return f"[도구 오류] {text}" if result.is_error else text


# 모델 없이 검색 도구만 직접 호출해 봅니다.
print(search_mcp_tools.invoke({"query": "프로젝트 예산 결재선"})[:500])

### 5.2 `wrap_tool_call` Middleware: 실행 직전의 정책 계층

`call_mcp_tool`은 이름만 주면 카탈로그의 어떤 도구든 실행할 수 있습니다. 그래서 **실행 직전에 "이 호출을 허용할지"를 판단하는 계층**이 필요합니다.

LangChain의 `wrap_tool_call` Middleware는 모델이 도구 호출을 결정한 뒤, 실제 도구가 실행되기 **직전**에 끼어듭니다.

- `handler(request)`를 호출하면 → 도구가 정상 실행됩니다.
- `handler`를 호출하지 않고 `ToolMessage`를 직접 반환하면 → 실행 없이 그 메시지가 도구 결과로 모델에게 전달됩니다.

여기서는 **"이번 대화에서 `search_mcp_tools`로 검색된 적 있는 도구만 실행 허용"** 규칙을 구현합니다. 모델이 스키마를 확인하지 않고 도구 이름을 추측해 호출하는 것을 막는 화이트리스트입니다.

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import ToolMessage


class DiscoveredToolGuard(AgentMiddleware):
    """call_mcp_tool이 '이번 대화에서 검색된 도구'만 실행하도록 막는 정책 계층"""

    def _block(self, request):
        call = request.tool_call
        if call["name"] != "call_mcp_tool":
            return None   # 다른 도구는 그대로 통과

        # 지금까지의 대화에서 search_mcp_tools가 반환한 도구 이름을 모읍니다.
        discovered = set()
        for m in request.state["messages"]:
            if isinstance(m, ToolMessage) and m.name == "search_mcp_tools":
                discovered |= {r["name"] for r in json.loads(m.content)["results"]}

        target = call["args"].get("name")
        if target in discovered:
            return None
        # 실행하지 않고, 모델에게 차단 사유를 도구 결과로 돌려줍니다.
        return ToolMessage(
            content=f"'{target}'은(는) 아직 검색되지 않은 도구입니다. search_mcp_tools로 먼저 검색하세요.",
            tool_call_id=call["id"],
            name=call["name"],
        )

    def wrap_tool_call(self, request, handler):
        return self._block(request) or handler(request)

    async def awrap_tool_call(self, request, handler):
        return self._block(request) or await handler(request)

### 5.3 에이전트 조립 및 실행

Part 4와 **같은 질문**을 던지고, 에이전트가 **검색 → 스키마 확인 → 실행**의 순서로 MCP 도구에 도달하는지 확인합니다.

In [ ]:
proxy_agent = create_agent(
    model=llm,
    tools=[search_mcp_tools, call_mcp_tool],   # 👈 MCP 도구가 몇 개든 바인딩은 이 2개로 고정
    system_prompt=(
        "당신은 사내 데이터 Q&A 에이전트입니다. "
        "필요한 도구는 search_mcp_tools로 먼저 찾고, call_mcp_tool로 실행하세요. "
        "도구 결과에 근거해서만 답하고 출처를 명시하세요."
    ),
    middleware=[DiscoveredToolGuard()],
    checkpointer=InMemorySaver(),
)

proxy_result = await ask(proxy_agent, COMPARE_QUESTION, "proxy-1")

In [ ]:
# 검색 결과(도구 스키마)가 대화 기록 '뒤쪽'에 ToolMessage로 덧붙었는지 확인합니다.
search_msgs = [m for m in proxy_result["messages"] if isinstance(m, ToolMessage) and m.name == "search_mcp_tools"]
for r in json.loads(search_msgs[0].content)["results"]:
    print(f"📎 {r['name']}\n   input_schema: {json.dumps(r['input_schema'], ensure_ascii=False)[:300]}\n")

### 💡 Search & Proxy 방식의 동작 정리

| 관점 | 동작 |
|:---|:---|
| **LLM에 바인딩된 도구** | `search_mcp_tools`, `call_mcp_tool` **2개로 고정**. 서버·도구가 늘어도 변하지 않습니다. |
| **도구 스키마가 들어오는 시점** | 모델이 검색한 순간, **검색 결과(ToolMessage)로 대화 뒤에 덧붙습니다.** prefix가 바뀌지 않으므로 프롬프트 캐싱이 유지됩니다. |
| **호출 형식** | 구조화된 JSON(`name`, `arguments`) 그대로 전달됩니다. shell 명령 문자열을 거치지 않습니다. |
| **인자 검증** | 범용 `arguments`이므로 provider 수준의 스키마 강제는 없고, **MCP 서버의 Pydantic 검증(Part 2)** 이 최종 방어선입니다. |
| **정책 계층** | `wrap_tool_call` Middleware가 실행 직전에 개입합니다. 화이트리스트 외에도 권한(scope) 확인, 감사 로그 등을 한곳에서 처리할 수 있습니다. |

> [!NOTE]
> **Provider 네이티브 Tool Search와의 관계**
> Claude API는 같은 아이디어를 API 기능으로 제공합니다(`defer_loading: true` + Tool Search Tool). 검색된 도구가 `tool_reference`로 대화에 펼쳐지고, 이후에는 정식 tool call로 호출되므로 스키마 강제까지 유지됩니다.
> 이번 Part의 Search & Proxy는 이 기능이 없는 모델(Gemini 등)에서도 동작하도록 **같은 원리를 애플리케이션 계층에서 구현**한 것입니다.

---
## 🧠 Part 6. 연결 방식 C: Anthropic Skills 기반 점진적 공개 (Progressive Disclosure)

Anthropic Agent Skills 패턴의 핵심 철학은 매우 단순합니다:
> *"에이전트에게 100개의 도구 스키마를 미리 쥐어주지 마라. 대신 **도구 사용 설명서(Skill 매뉴얼)**와 **3개의 범용 원시 도구(Primitive Tools)**만 주어라."*

Skills 방식에서 에이전트는 비즈니스 도구의 존재를 미리 알지 못합니다. 파일 시스템을 탐색하고 터미널 명령을 내릴 수 있는 기본 도구 3종(`glob_search`, `file_read`, `bash_command`)을 장착합니다.
Part 5가 **전용 검색·호출 도구**로 MCP에 접근했다면, Skills는 **파일 읽기와 CLI 실행이라는 더 범용적인 도구**로 같은 일을 합니다.

```mermaid
flowchart TD
    subgraph Agent ["(create_agent)"]
        Tools["범용 원시 도구 (tools/)<br/>• glob_search<br/>• file_read<br/>• bash_command"]
    end

    subgraph LocalContext ["로컬 파일시스템 (app/agents/skill_rag_agent/skills/)"]
        Cat["1. 스킬 카탈로그 <br/> (매 턴 프롬프트 주입) <br/>• mcp: 원격 MCP 서버 CLI 연동 도구"]
        SkillMD["2. skills/mcp/Skill.md<br/>• CLI 사용법 및 옵션 설명서"]
        MCPRegistry["3. skills/mcp/references/mcp_servers.json <br/>• 사내 MCP 서버 레지스트리"]
    end

    subgraph RemoteServer ["원격 마이크로서비스"]
        MCPServer["4. Enterprise-RAG-Server (:8010)<br/>• list_tools.py (도구 목록 조회)<br/>• execute_tool.py (도구 실행)"]
    end

    Agent -->|"1. 스킬 필요 판단"| Cat
    Agent -->|"2. file_read (매뉴얼 학습)"| SkillMD
    Agent -->|"3. file_read (서버 정보 확인)"| MCPRegistry
    Agent -->|"4. bash_command (CLI 실행)"| MCPServer
```

### 6.1 점진적 공개 (Progressive Disclosure)의 3단계 라이프사이클

| 단계 | 정보 | 언제 컨텍스트에 들어오는가? | 토큰 비용 |
|:---:|:---|:---|:---:|
| **1단계: 식별 (Catalog)** | 스킬 이름과 한 줄 요약 | **항상** (시스템 프롬프트 상단) | 수십 토큰 |
| **2단계: 학습 (Instruction)** | 스킬 매뉴얼 (`Skill.md`) 전문 | **해당 스킬이 필요하다고 판단된 순간** (`file_read`) | 수백 토큰 |
| **3단계: 실행 (Execution)** | 서버의 도구 스키마 및 실행 결과 | **실제 명령을 실행하는 순간** (`bash_command`) | 호출 시점만 |

이제 에이전트의 프롬프트에 매 턴 주입되는 **스킬 카탈로그(`assemble()`)**와, 서버 목록이 적힌 **MCP 레지스트리**를 직접 열어봅니다.

In [ ]:
from app.agents.skill_rag_agent.prompt import BASE_SKILL_RAG_SYSTEM_PROMPT, get_skill_prompt_builder

builder = get_skill_prompt_builder()
print("=== 매 호출마다 SkillCatalogMiddleware가 프롬프트에 주입하는 블록 ===")
print(builder.assemble())

In [ ]:
import json

# 💡 아래 셀을 실행하기 전에 missions/03_missions.md의 1~2단계(CLI 보안 검증 및 mcp_servers.json 등록)를 완료하세요.
registry_text = open("app/agents/skill_rag_agent/skills/mcp/references/mcp_servers.json", encoding="utf-8").read()
print(registry_text)
if "enterprise_rag" not in json.loads(registry_text):
    print("⚠️ enterprise_rag가 등록되어 있지 않습니다. Mission 03(missions/03_missions.md)의 2단계(레지스트리 등록)를 먼저 완료하세요.")

> [!IMPORTANT]
> ### 🔒 엔터프라이즈 보안: `--token-env` 간접 주입 패턴
> 
> 왜 레지스트리(`app/agents/skill_rag_agent/skills/mcp/references/mcp_servers.json`)에 실제 토큰 문자열이 아니라 **환경 변수 이름(`ENTERPRISE_RAG_MCP_TOKEN`)** 만 적혀 있을까요?
> 
> 1. **프롬프트 · 로그 유출 방지**: 레지스트리와 매뉴얼은 에이전트가 읽어 대화 기록에 들어갑니다. 여기에 토큰 값이 없으므로 대화 기록, LangSmith 같은 트레이스, 서버 로그에 토큰이 평문으로 남지 않습니다.
> 2. **실제 토큰 사용**: CLI 스크립트(`app/agents/skill_rag_agent/skills/mcp/scripts/execute_tool.py`)가 실행 시점에 OS 환경 변수에서 토큰을 읽어 `Authorization: Bearer <토큰>` 헤더로 전송합니다.
>
> ⚠️ **한계: shell을 가진 에이전트에게 환경 변수는 보안 경계가 아닙니다.**
> Skills 에이전트는 `bash_command`를 갖고 있으므로, 프롬프트 인젝션으로 `printenv`나 `echo $ENTERPRISE_RAG_MCP_TOKEN`을 실행하게 만들면 토큰이 그대로 노출됩니다.
> 운영 환경에서는 에이전트를 sandbox에서 실행하고, 토큰은 **sandbox 밖의 별도 프로세스(자격 증명 브로커, 프록시)** 가 보관해 헤더에 붙이도록 분리합니다.
> 참고로 Part 5의 Search & Proxy 방식은 토큰이 애플리케이션 프로세스(`call_mcp_tool`) 안에만 있고 에이전트에게 shell이 없으므로 이 문제가 생기지 않습니다.

---

### 6.2 CLI로 직접 호출해 보기 (에이전트가 수행할 동작을 사람이 먼저 검증)

에이전트에게 맡기기 전에, 에이전트가 실행하게 될 CLI 명령어(`list_tools.py`, `execute_tool.py`)를 개발자가 직접 `bash_command`로 호출하여 정상 동작하는지 확인합니다.

In [ ]:
from app.tools import bash_command

print(bash_command.invoke({"command": "python app/agents/skill_rag_agent/skills/mcp/scripts/list_tools.py --url http://localhost:8010/mcp --token-env ENTERPRISE_RAG_MCP_TOKEN"})[:900])

In [ ]:
cmd = (
    "python app/agents/skill_rag_agent/skills/mcp/scripts/execute_tool.py --url http://localhost:8010/mcp --token-env ENTERPRISE_RAG_MCP_TOKEN "
    "--tool graph_neighbors --args '{\"entity\": \"AGENTIC AI TEAM\", \"query\": \"총괄 프로젝트와 보고 라인\", \"top_k\": 5}'"
)
print(bash_command.invoke({"command": cmd}))

### 6.3 Skills 에이전트 조립 및 실행

이제 `app/agents/skill_rag_agent/`에 구현된 구조 그대로 에이전트를 조립합니다.
에이전트에게는 비즈니스 도구를 주지 않고, **범용 도구 3종(`glob_search`, `file_read`, `bash_command`)**과 매 턴 스킬 카탈로그를 프롬프트에 주입해 주는 **`SkillCatalogMiddleware`**만을 장착합니다.

Part 4·5와 **같은 질문**을 던지고, 에이전트가 스스로 **설명서 읽기 → 서버 레지스트리 확인 → 도구 목록 조회 → 도구 실행**의 점진적 탐색 루프를 거치는지 확인합니다.

In [ ]:
from app.tools import glob_search, file_read, bash_command
from app.middleware import SkillCatalogMiddleware

skill_agent = create_agent(
    model=llm,
    tools=[glob_search, file_read, bash_command],
    system_prompt=BASE_SKILL_RAG_SYSTEM_PROMPT,
    middleware=[SkillCatalogMiddleware(builder)],
    checkpointer=InMemorySaver(),
)

result = await ask(skill_agent, COMPARE_QUESTION, "skill-1")

In [ ]:
# 토큰이 대화 기록 어디에도 노출되지 않았는지 확인
leaked = any(TOKEN in str(m.content) or TOKEN in str(getattr(m, "tool_calls", "")) for m in result["messages"])
print("🔒 토큰 노출 여부:", "노출됨 ❌" if leaked else "노출 없음 ✅")

### 💡 세 연결 방식의 비교 (Trade-off Analysis)

같은 질문을 세 에이전트에 던졌을 때, 위 실행 결과의 **도구 호출 경로(Step)** 를 나란히 놓고 비교해 보세요.

| 비교 지표 | A: 정적 바인딩 | B: Search & Proxy | C: Skills |
|:---|:---|:---|:---|
| **LLM에 바인딩되는 도구** | MCP 도구 **전부** | **2개 고정** (`search_mcp_tools`, `call_mcp_tool`) | **3개 고정** (`glob_search`, `file_read`, `bash_command`) |
| **도구 스키마가 컨텍스트에 들어오는 시점** | 매 턴 **전부** | 검색한 도구만, **검색 결과로 대화 뒤에** | 매뉴얼을 읽고 `list_tools`를 실행했을 때, **도구 결과로 대화 뒤에** |
| **호출 경로** | LLM → MCP 도구 | LLM → `call_mcp_tool` → Middleware 검사 → MCP | LLM → `bash_command` → CLI 스크립트 → MCP |
| **인자 전달 형식** | 구조화된 JSON (도구별 스키마 강제) | 구조화된 JSON (범용 `arguments`, 서버 Pydantic이 검증) | **shell 명령 문자열** 안에 JSON (따옴표 이스케이프 필요) |
| **탐색 단계 (호출 횟수 · 지연)** | ✅ 최소: 바로 호출 | ⚠️ +1: 검색 후 호출 | ⚠️ +2\~3: 매뉴얼 · 레지스트리 · 목록 확인 후 호출 |
| **도구 수 확장성** | ❌ 도구가 늘수록 컨텍스트 점유 · 오선택 증가 | ✅ 카탈로그 크기와 무관하게 프롬프트 고정 | ✅ 카탈로그 크기와 무관하게 프롬프트 고정 |
| **실행 환경 요구** | 없음 | 없음 | **shell 실행 권한** (실무에서는 sandbox 필요) |
| **실무 적합 영역** | 도구가 적고 고정된 목적형 에이전트 | 도구가 많은 일반 서비스 에이전트 | shell · sandbox가 원래 있는 코딩 · 분석 에이전트 |

> **아키텍처 인사이트: 세 방식 모두 `tools`는 고정되어 있습니다.**
> A는 처음부터 전부 넣어서 고정이고, B와 C는 범용 도구만 넣어서 고정입니다. 세 방식 모두 대화 도중 `tools`를 바꾸지 않으므로 **프롬프트 캐싱이 유지**됩니다.
> 차이는 **"도구 정보를 언제, 어떤 경로로 모델에게 보여주는가"** 입니다. A는 미리 전부, B는 검색 결과로, C는 매뉴얼과 CLI 출력으로 보여줍니다.
>
> 도구가 적으면 A가 가장 단순하고 빠릅니다. 도구가 많아지면 B나 C로 넘어가는데, 일반 서비스라면 shell이 필요 없는 B가, 이미 shell과 sandbox를 갖춘 에이전트라면 매뉴얼로 사용법까지 전달할 수 있는 C가 자연스럽습니다.

### ⚖️ 아키텍처별 장단점 한눈에 보기

| 아키텍처 | 👍 장점 | 👎 단점 | 🎯 이럴 때 선택 |
|:---|:---|:---|:---|
| **A. 정적 바인딩**<br>(`MultiServerMCPClient`) | • 구현이 가장 단순함 (어댑터 몇 줄)<br>• 도구별 스키마가 provider 수준에서 강제되어 **호출 안정성 최상**<br>• 탐색 단계가 없어 **호출 횟수 · 지연 최소** | • 도구가 늘수록 **모든 스키마가 매 턴 컨텍스트를 점유**<br>• 후보가 많아지면 **비슷한 도구를 잘못 고르는 확률 증가**<br>• 도구 구성이 에이전트 시작 시점에 결정됨 | 도구가 적고(\~20개) 구성이 자주 바뀌지 않는 **목적형 에이전트** |
| **B. Search & Proxy**<br>(`search_mcp_tools` + `call_mcp_tool` + Middleware) | • 도구 수와 무관하게 **바인딩은 2개로 고정**, 필요한 스키마만 컨텍스트에 들어옴<br>• 인자가 **구조화된 JSON**으로 전달되어 shell 관련 문제가 없음<br>• `wrap_tool_call`로 **화이트리스트 · 권한 · 감사 로그를 한곳에서** 통제<br>• provider 기능에 의존하지 않음 | • 검색 단계만큼 **호출이 1회 이상 늘어남**<br>• 범용 `arguments`라서 provider 수준 **스키마 강제가 없음** (서버 검증에 의존)<br>• 검색 품질이 낮으면 필요한 도구를 못 찾음 | 도구가 많은 **일반 서비스 에이전트** (사내 챗봇, 업무 지원 에이전트 등) |
| **C. Skills**<br>(`file_read` + `bash_command` + 스킬 카탈로그) | • 도구 수와 무관하게 **바인딩은 3개로 고정**<br>• 스키마뿐 아니라 **사용법 · 주의사항 · 절차까지 매뉴얼로 전달** 가능<br>• 같은 범용 도구로 **MCP 외의 작업**(파일 처리, 스크립트 실행)까지 확장 | • 매뉴얼 · 레지스트리 · 목록 확인으로 **탐색 단계가 가장 많음**<br>• 인자를 shell 명령 문자열에 넣어야 해서 **따옴표 이스케이프 실패 위험**<br>• **shell 실행 권한**이 필요해 sandbox 등 보안 장치가 필수 | shell과 sandbox가 원래 있는 **코딩 · 데이터 분석 에이전트** |

> 💡 **한 줄 요약**
> 도구가 적으면 **A**, 많으면 **B**, 에이전트가 이미 shell을 다루는 환경이라면 **C**.
> 세 방식 모두 `tools`를 대화 도중 바꾸지 않으므로 프롬프트 캐싱은 공통으로 유지됩니다.

---
## 🏁 Part 7. 정리와 Mission 03

이번 시간에는 고립된 RAG 도구를 엔터프라이즈 표준 규격인 **FastMCP 서버**로 격리 구축하고,
같은 MCP 서버를 **정적 바인딩**, **Search & Proxy**, **Skills(점진적 공개)** 세 가지 방식으로 에이전트에 연결해 보며, 도구 정보를 모델에게 전달하는 경로의 차이와 트레이드오프를 비교했습니다.

실습을 마치기 전, 노트북에서 백그라운드로 띄웠던 테스트용 MCP 서버 프로세스를 안전하게 종료합니다.

In [ ]:
# 노트북에서 띄운 MCP 서버 종료 (이미 떠 있던 서버는 그대로 둡니다)
if mcp_process is not None:
    mcp_process.terminate()
    mcp_process.wait(timeout=10)
    print("🛑 MCP 서버 종료")

### 🎯 Mission 03. Skills 에이전트 레지스트리 연동 및 Tool vs Skills 비교

`missions/03_missions.md`의 안내에 따라 실습을 마무리하세요.

1. **CLI 보안 검증 및 레지스트리 등록 (1\~2단계)**: `list_tools.py` / `execute_tool.py`로 인증 동작을 확인하고 `mcp_servers.json`에 `enterprise_rag`를 등록합니다.
2. **FastMCP 서버 실행 확인 (3단계)**: 위 셀에서 노트북이 띄운 서버를 종료했다면, 터미널에서 `python app/mcp/enterprise_rag_server.py --port 8010`으로 다시 띄웁니다.
3. **Chainlit UI에서 두 에이전트 대화 비교**:
   - `tool_rag_agent`: 1교시의 도구 직접 바인딩 에이전트
   - `skill_rag_agent`: 이번 시간에 배운 Skills + MCP 자율 탐색 에이전트
   - 두 에이전트에게 동일한 질문을 던지고, Chainlit 단계별 로그(Step)에서 도구 호출 패턴과 토큰 사용량의 차이를 직접 눈으로 확인합니다.